In [ ]:
import polars as pl

DATA = "../data/raw/tweets.csv"

In [ ]:
raw_df = pl.read_csv(DATA)
df = raw_df.with_columns(
    (pl.col("isRetweet").str.to_lowercase() == "t").alias("is_retweet"),
    (pl.col("isDeleted").str.to_lowercase() == "t").alias("is_deleted"),
    (pl.col("isFlagged").str.to_lowercase() == "t").alias("is_flagged"),
    pl.col("date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    (pl.col("retweets") + 1).log().alias("log_retweets"),
    (pl.col("favorites") + 1).log().alias("log_favorites"),
    pl.col("device").cast(pl.Categorical),
    pl.col("text")
    .str.to_lowercase()
    .alias("text_lower"),
    pl.col("text_lower")
    .str.replace_all(r"[^\w\s]", "")
    .alias("text_lower_no-punctuation")
).drop(["isRetweet", "isDeleted", "isFlagged"])
df

In [ ]:
col = "device"
df[col].value_counts().sort(by="count", descending=True)

In [ ]:
(df["retweets"] + 1).log().describe()

In [ ]:
df.write_parquet("../data/processed/trump_parsed.parquet")